# Clase 10 — Llevar modelos a producción con TensorFlow Extended (TFX)

## Objetivos de esta clase

- Describir el ciclo de vida de un modelo de Machine Learning (ML) en producción y los componentes de un pipeline de TensorFlow Extended (TFX)
- Construir un pipeline simple de TFX de principio a fin
- Validar datos, transformar features y analizar el modelo con TensorFlow Data Validation (TFDV), TensorFlow Transform (TFT) y TensorFlow Model Analysis (TFMA)

**Antes de empezar:** este notebook instala el paquete `tfx`, y esa instalación puede tardar varios minutos en Google Colab porque arrastra muchas dependencias (Apache Beam, TFDV, TFT, TFMA...). Es normal ver un aluvión de mensajes de `pip`. Si Colab te pide reiniciar el entorno de ejecución al terminar, hazlo y continúa desde la celda de comprobación de versiones.


## De notebook a producción

Hasta ahora entrenabas un modelo llamando a `fit` en una celda y revisando el resultado tú mismo. Eso no escala a un sistema real: en producción necesitas ingerir datos nuevos, validarlos, entrenar, evaluar y desplegar el modelo de forma automática y repetible, sin que nadie tenga que reejecutar un notebook a mano cada vez.

**TensorFlow Extended (TFX)** es la plataforma de código abierto de Google para resolver justo eso: ingeniería de Machine Learning (ML) para producción. Es la plataforma que usa Google internamente para la mayoría de sus sistemas de ML, y también la usan empresas como Twitter, Airbnb o PayPal.

TFX no sustituye a TensorFlow (TF): se apoya en él y añade bibliotecas especializadas para cada fase del ciclo de vida de un modelo:

- **TensorFlow Data Validation (TFDV)** — validación de datos
- **TensorFlow Transform (TFT)** — transformación de features
- **TensorFlow Model Analysis (TFMA)** — análisis del modelo entrenado
- **TensorFlow Serving** — despliegue del modelo como servicio


## Qué es un pipeline de TFX

Un **pipeline** de TFX es una secuencia de **componentes** organizados como un grafo acíclico dirigido (DAG, Directed Acyclic Graph): los datos fluyen siempre hacia delante, nunca hacia atrás. Cada componente recibe artefactos de entrada (datos, esquemas, modelos), hace una tarea concreta y produce artefactos de salida que quedan disponibles para el siguiente componente.

Internamente cada componente tiene tres partes — un *driver* (recupera entradas), un *executor* (hace el trabajo real) y un *publisher* (registra las salidas) — pero como usuario casi siempre trabajas solo con el executor, normalmente escribiendo una función Python que TFX llama por ti.

| Componente | Qué hace |
|---|---|
| `ExampleGen` | Ingiere los datos y los divide en entrenamiento/evaluación |
| `StatisticsGen` | Calcula estadísticas descriptivas por feature |
| `SchemaGen` | Infiere un esquema de tipos y rangos válidos |
| `ExampleValidator` | Detecta anomalías respecto al esquema |
| `Transform` | Aplica ingeniería de features (TFT) |
| `Trainer` | Entrena el modelo con TensorFlow |
| `Evaluator` | Analiza el rendimiento del modelo (TFMA), incluso por segmentos |
| `Pusher` | Publica el modelo hacia un destino de despliegue |

Hoy vas a construir el pipeline mínimo — `ExampleGen` + `Trainer` + `Pusher` — y después verás, a nivel práctico, qué añaden TFDV, TFT y TFMA sobre esa base.


In [ ]:
# La instalación de tfx puede tardar varios minutos: instala Apache Beam,
# TensorFlow Data Validation, TensorFlow Transform y TensorFlow Model Analysis,
# entre otras dependencias. Ten paciencia la primera vez.
try:
    import colab
    IN_COLAB = True
    !pip install --upgrade pip -q
except ImportError:
    IN_COLAB = False

!pip install -U tfx -q


Si estás en Colab, es posible que tengas que reiniciar el entorno de ejecución (menú **Entorno de ejecución > Reiniciar sesión**) después de esta instalación, antes de continuar. Es un comportamiento normal la primera vez que instalas `tfx` en una sesión.


In [ ]:
import os
import shutil
import tempfile
import urllib.request

import tensorflow as tf
from tfx import v1 as tfx
from absl import logging

logging.set_verbosity(logging.INFO)

print('TensorFlow version:', tf.__version__)
print('TFX version:', tfx.__version__)


## Preparamos los datos: Palmer Penguins

Usaremos el conjunto de datos [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/articles/intro.html), ya normalizado entre 0 y 1, con cuatro features numéricos (longitud y profundidad del pico, longitud de la aleta y masa corporal) para predecir la especie del pingüino. `ExampleGen` necesita leer los datos de un directorio, así que descargamos el CSV a una carpeta temporal.


In [ ]:
PIPELINE_NAME = "penguin-simple"
PIPELINE_ROOT = os.path.join('pipelines', PIPELINE_NAME)
METADATA_PATH = os.path.join('metadata', PIPELINE_NAME, 'metadata.db')
SERVING_MODEL_DIR = os.path.join('serving_model', PIPELINE_NAME)

DATA_ROOT = tempfile.mkdtemp(prefix='tfx-data')
_data_url = ('https://raw.githubusercontent.com/tensorflow/tfx/master/tfx/'
             'examples/penguin/data/labelled/penguins_processed.csv')
_data_filepath = os.path.join(DATA_ROOT, "data.csv")
urllib.request.urlretrieve(_data_url, _data_filepath)

with open(_data_filepath) as f:
    for _ in range(5):
        print(f.readline().strip())


`species` vale 0, 1 o 2 (una de las tres especies) y el resto de columnas ya están entre 0 y 1.

## Escribimos el código de entrenamiento

TFX usa el **Trainer genérico**: le pasas un archivo Python con una función `run_fn`, que es el punto de entrada que el componente `Trainer` llamará por ti. Dentro definimos también cómo leer los datos y cómo construir el modelo Keras.


In [ ]:
_trainer_module_file = 'penguin_trainer.py'


In [ ]:
%%writefile {_trainer_module_file}

from typing import List
from absl import logging
import tensorflow as tf
from tensorflow import keras
from tensorflow_transform.tf_metadata import schema_utils

from tfx import v1 as tfx
from tfx_bsl.public import tfxio
from tensorflow_metadata.proto.v0 import schema_pb2

_FEATURE_KEYS = [
    'culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g'
]
_LABEL_KEY = 'species'

_TRAIN_BATCH_SIZE = 20
_EVAL_BATCH_SIZE = 10

# Como todavia no usamos un esquema generado por SchemaGen, describimos
# manualmente el formato esperado de cada feature.
_FEATURE_SPEC = {
    **{
        feature: tf.io.FixedLenFeature(shape=[1], dtype=tf.float32)
        for feature in _FEATURE_KEYS
    },
    _LABEL_KEY: tf.io.FixedLenFeature(shape=[1], dtype=tf.int64)
}


def _input_fn(file_pattern: List[str],
              data_accessor: tfx.components.DataAccessor,
              schema: schema_pb2.Schema,
              batch_size: int = 200) -> tf.data.Dataset:
  """Convierte los TFRecords que produce ExampleGen en un tf.data.Dataset."""
  return data_accessor.tf_dataset_factory(
      file_pattern,
      tfxio.TensorFlowDatasetOptions(
          batch_size=batch_size, label_key=_LABEL_KEY),
      schema=schema).repeat()


def _build_keras_model() -> tf.keras.Model:
  """Crea una red densa (DNN) sencilla para clasificar pinguinos."""
  inputs = [keras.layers.Input(shape=(1,), name=f) for f in _FEATURE_KEYS]
  d = keras.layers.concatenate(inputs)
  for _ in range(2):
    d = keras.layers.Dense(8, activation='relu')(d)
  outputs = keras.layers.Dense(3)(d)

  model = keras.Model(inputs=inputs, outputs=outputs)
  model.compile(
      optimizer=keras.optimizers.Adam(1e-2),
      loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
      metrics=[keras.metrics.SparseCategoricalAccuracy()])

  model.summary(print_fn=logging.info)
  return model


# TFX llama a esta funcion: es el punto de entrada del componente Trainer.
def run_fn(fn_args: tfx.components.FnArgs):
  schema = schema_utils.schema_from_feature_spec(_FEATURE_SPEC)

  train_dataset = _input_fn(
      fn_args.train_files, fn_args.data_accessor, schema,
      batch_size=_TRAIN_BATCH_SIZE)
  eval_dataset = _input_fn(
      fn_args.eval_files, fn_args.data_accessor, schema,
      batch_size=_EVAL_BATCH_SIZE)

  model = _build_keras_model()
  model.fit(
      train_dataset,
      steps_per_epoch=fn_args.train_steps,
      validation_data=eval_dataset,
      validation_steps=fn_args.eval_steps)

  # TFX espera encontrar el modelo guardado exactamente en esta ruta.
  model.save(fn_args.serving_model_dir, save_format='tf')


## Definimos y ejecutamos el pipeline

Ahora encadenamos los tres componentes: `CsvExampleGen` ingiere el CSV, `Trainer` entrena el modelo con el código de arriba, y `Pusher` copia el modelo entrenado a una carpeta de destino. Fíjate en que nunca llamamos a `fit` directamente aquí: describimos qué componentes existen y de dónde toma cada uno sus entradas, y dejamos que TFX resuelva el orden de ejecución.


In [ ]:
def _create_pipeline(pipeline_name: str, pipeline_root: str, data_root: str,
                      module_file: str, serving_model_dir: str,
                      metadata_path: str) -> tfx.dsl.Pipeline:
  """Pipeline de tres componentes: ExampleGen, Trainer, Pusher."""
  # 1. Ingiere el CSV y lo divide en entrenamiento/evaluacion.
  example_gen = tfx.components.CsvExampleGen(input_base=data_root)

  # 2. Entrena el modelo con el codigo de penguin_trainer.py.
  trainer = tfx.components.Trainer(
      module_file=module_file,
      examples=example_gen.outputs['examples'],
      train_args=tfx.proto.TrainArgs(num_steps=100),
      eval_args=tfx.proto.EvalArgs(num_steps=5))

  # 3. Publica el modelo entrenado en el sistema de archivos.
  pusher = tfx.components.Pusher(
      model=trainer.outputs['model'],
      push_destination=tfx.proto.PushDestination(
          filesystem=tfx.proto.PushDestination.Filesystem(
              base_directory=serving_model_dir)))

  components = [example_gen, trainer, pusher]

  return tfx.dsl.Pipeline(
      pipeline_name=pipeline_name,
      pipeline_root=pipeline_root,
      metadata_connection_config=tfx.orchestration.metadata
      .sqlite_metadata_connection_config(metadata_path),
      components=components)


In [ ]:
# LocalDagRunner ejecuta el pipeline directamente en esta maquina; es ideal
# para desarrollo. En produccion se usan orquestadores como Apache Airflow
# o Kubeflow Pipelines.
tfx.orchestration.LocalDagRunner().run(
    _create_pipeline(
        pipeline_name=PIPELINE_NAME,
        pipeline_root=PIPELINE_ROOT,
        data_root=DATA_ROOT,
        module_file=_trainer_module_file,
        serving_model_dir=SERVING_MODEL_DIR,
        metadata_path=METADATA_PATH))


Si todo fue bien, deberías ver `INFO:absl:Component Pusher is finished.` al final del registro (log): es el último componente del pipeline. Comprobamos que el modelo se guardó donde esperábamos:


In [ ]:
!find {SERVING_MODEL_DIR}


## Qué añade TensorFlow Data Validation (TFDV)

Antes de fiarte de un modelo, tienes que fiarte de los datos. TFDV añade tres componentes al pipeline:

- `StatisticsGen` hace una pasada completa sobre los datos y calcula estadísticas por feature (mínimo, máximo, valores ausentes, histogramas)
- `SchemaGen` examina esas estadísticas y genera automáticamente un **esquema**: tipos de dato, si cada feature es obligatoria, qué categorías son válidas
- `ExampleValidator` compara datos nuevos contra ese esquema y reporta anomalías

Vamos a construir una versión reducida de este flujo: un pipeline de solo dos pasos (`StatisticsGen` + `SchemaGen`) sobre nuestros mismos datos, para generar y revisar el esquema.


In [ ]:
SCHEMA_PIPELINE_NAME = "penguin-schema"
SCHEMA_PIPELINE_ROOT = os.path.join('pipelines', SCHEMA_PIPELINE_NAME)
SCHEMA_METADATA_PATH = os.path.join('metadata', SCHEMA_PIPELINE_NAME, 'metadata.db')


def _create_schema_pipeline(pipeline_name, pipeline_root, data_root, metadata_path):
  """Pipeline corto solo para generar y revisar el esquema de los datos."""
  example_gen = tfx.components.CsvExampleGen(input_base=data_root)

  # NUEVO: calcula estadisticas descriptivas de cada feature.
  statistics_gen = tfx.components.StatisticsGen(
      examples=example_gen.outputs['examples'])

  # NUEVO: infiere un esquema (tipos, rangos) a partir de esas estadisticas.
  schema_gen = tfx.components.SchemaGen(
      statistics=statistics_gen.outputs['statistics'],
      infer_feature_shape=True)

  components = [example_gen, statistics_gen, schema_gen]

  return tfx.dsl.Pipeline(
      pipeline_name=pipeline_name,
      pipeline_root=pipeline_root,
      metadata_connection_config=tfx.orchestration.metadata
      .sqlite_metadata_connection_config(metadata_path),
      components=components)


tfx.orchestration.LocalDagRunner().run(
    _create_schema_pipeline(
        pipeline_name=SCHEMA_PIPELINE_NAME,
        pipeline_root=SCHEMA_PIPELINE_ROOT,
        data_root=DATA_ROOT,
        metadata_path=SCHEMA_METADATA_PATH))


In [ ]:
# Localizamos el esquema generado a traves del metadata del pipeline
# y lo copiamos a una carpeta fija, como harias antes de revisarlo a mano.
from tfx.orchestration.metadata import Metadata
from tfx.types import standard_component_specs
from tfx.orchestration.portable.mlmd import execution_lib


def get_latest_artifacts(metadata, pipeline_name, component_id):
  context = metadata.store.get_context_by_type_and_name(
      'node', f'{pipeline_name}.{component_id}')
  executions = metadata.store.get_executions_by_context(context.id)
  latest_execution = max(executions, key=lambda e: e.last_update_time_since_epoch)
  return execution_lib.get_output_artifacts(metadata, latest_execution.id)


metadata_connection_config = tfx.orchestration.metadata.sqlite_metadata_connection_config(
    SCHEMA_METADATA_PATH)

with Metadata(metadata_connection_config) as metadata_handler:
  schema_gen_output = get_latest_artifacts(
      metadata_handler, SCHEMA_PIPELINE_NAME, 'SchemaGen')
  schema_artifacts = schema_gen_output[standard_component_specs.SCHEMA_KEY]

SCHEMA_PATH = 'schema'
os.makedirs(SCHEMA_PATH, exist_ok=True)
_generated_path = os.path.join(schema_artifacts[0].uri, 'schema.pbtxt')
shutil.copy(_generated_path, SCHEMA_PATH)

print(open(os.path.join(SCHEMA_PATH, 'schema.pbtxt')).read())


Deberías ver cuatro features `FLOAT` y una `species` de tipo `INT`. Con este esquema ya revisado, el paso siguiente en un pipeline real sería importarlo con un componente `Importer` y añadir `ExampleValidator` justo antes del `Trainer`, así:

```python
# Ilustrativo: así se vería el pipeline de entrenamiento con validación.
# No lo ejecutamos aquí para no repetir el entrenamiento del modelo.
schema_importer = tfx.dsl.Importer(
    source_uri=SCHEMA_PATH,
    artifact_type=tfx.types.standard_artifacts.Schema
).with_id('schema_importer')

example_validator = tfx.components.ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_importer.outputs['result'])

# El Trainer también recibiría el esquema, en vez del _FEATURE_SPEC manual:
trainer = tfx.components.Trainer(
    module_file=module_file,
    examples=example_gen.outputs['examples'],
    schema=schema_importer.outputs['result'],
    train_args=tfx.proto.TrainArgs(num_steps=100),
    eval_args=tfx.proto.EvalArgs(num_steps=5))
```

Por defecto, las anomalías que detecta `ExampleValidator` no bloquean el pipeline: quedan registradas para que una persona decida qué hacer.


## Qué añade TensorFlow Transform (TFT)

Nuestros features ya venían normalizados entre 0 y 1. En un caso real casi nunca tienes esa suerte. TFT resuelve la ingeniería de features con una idea central: **el mismo código de transformación se aplica en entrenamiento y en servicio**, así evitas el sesgo entrenamiento-servicio (training/serving skew).

La pieza clave es la función `preprocessing_fn`. Dentro puedes usar operaciones normales de TensorFlow (se aplican ejemplo a ejemplo) o **analizadores de TFT** como `tft.mean` o `tft.scale_to_z_score`, que hacen una pasada completa sobre todo el dataset para calcular una constante (una media, una varianza) y la incrustan en el grafo. Vamos a ver una demo mínima, standalone, con datos de juguete, para entender la mecánica antes de imaginarla dentro de un componente `Transform`.


In [ ]:
import pprint
import tensorflow_transform as tft
import tensorflow_transform.beam as tft_beam
from tensorflow_transform.tf_metadata import dataset_metadata, schema_utils

raw_data = [
    {'x': 1, 'y': 1, 's': 'hello'},
    {'x': 2, 'y': 2, 's': 'world'},
    {'x': 3, 'y': 3, 's': 'hello'},
]

raw_data_metadata = dataset_metadata.DatasetMetadata(
    schema_utils.schema_from_feature_spec({
        'y': tf.io.FixedLenFeature([], tf.float32),
        'x': tf.io.FixedLenFeature([], tf.float32),
        's': tf.io.FixedLenFeature([], tf.string),
    }))


def preprocessing_fn(inputs):
  """Funcion que TFT ejecutara igual en entrenamiento y en servicio."""
  x, y, s = inputs['x'], inputs['y'], inputs['s']
  # tft.mean() hace una pasada completa sobre TODO el dataset; no es una
  # operacion por ejemplo, a diferencia de una resta normal de TensorFlow.
  x_centered = x - tft.mean(x)
  y_normalized = tft.scale_to_0_1(y)
  # Convierte cadenas de texto en indices segun su frecuencia.
  s_integerized = tft.compute_and_apply_vocabulary(s)
  return {
      'x_centered': x_centered,
      'y_normalized': y_normalized,
      's_integerized': s_integerized,
  }


with tft_beam.Context(temp_dir=tempfile.mkdtemp()):
  transformed_dataset, _ = (
      (raw_data, raw_data_metadata)
      | tft_beam.AnalyzeAndTransformDataset(preprocessing_fn))

transformed_data, _ = transformed_dataset
print('Datos originales:')
pprint.pprint(raw_data)
print('Datos transformados:')
pprint.pprint(transformed_data)


Fíjate en que `x_centered` queda centrado en torno a 0 usando la media de **todo** el dataset (2), `y_normalized` queda escalado entre 0 y 1, y `s_integerized` convierte `'hello'`/`'world'` en índices enteros. Dentro de un pipeline de TFX, esta misma `preprocessing_fn` se pasa al componente `Transform`, que produce un `transform_graph` que el `Trainer` incorpora al modelo final:

```python
# Ilustrativo: así se integraría Transform en el pipeline de penguins.
# No lo ejecutamos aquí para no repetir el entrenamiento.
transform = tfx.components.Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_importer.outputs['result'],
    module_file=module_file)  # aquí viviría preprocessing_fn

trainer = tfx.components.Trainer(
    module_file=module_file,
    examples=example_gen.outputs['examples'],
    transform_graph=transform.outputs['transform_graph'],  # NUEVO
    train_args=tfx.proto.TrainArgs(num_steps=100),
    eval_args=tfx.proto.EvalArgs(num_steps=5))
```

Así, cuando el modelo ya está sirviendo predicciones en producción, aplica exactamente las mismas transformaciones que usó durante el entrenamiento.


## Qué añade TensorFlow Model Analysis (TFMA)

Un modelo puede tener una precisión (accuracy) global aceptable y aun así fallar sistemáticamente en un segmento de datos que a tu negocio le importa mucho. TFMA añade el componente `Evaluator`, que analiza el rendimiento del modelo por **segmentos** (slicing) y decide si el modelo nuevo debe "bendecirse" (blessed) para producción, comparándolo con el modelo anterior.

```python
# Ilustrativo: así se integraría Evaluator en el pipeline de penguins.
# No lo ejecutamos aquí porque requiere reentrenar y comparar versiones.
import tensorflow_model_analysis as tfma

model_resolver = tfx.dsl.Resolver(
    strategy_class=tfx.dsl.experimental.LatestBlessedModelStrategy,
    model=tfx.dsl.Channel(type=tfx.types.standard_artifacts.Model),
    model_blessing=tfx.dsl.Channel(type=tfx.types.standard_artifacts.ModelBlessing)
).with_id('latest_blessed_model_resolver')

eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key='species')],
    slicing_specs=[
        tfma.SlicingSpec(),                          # el dataset completo
        tfma.SlicingSpec(feature_keys=['species']),  # cada especie por separado
    ],
    metrics_specs=[
        tfma.MetricsSpec(per_slice_thresholds={
            'sparse_categorical_accuracy': tfma.PerSliceMetricThresholds(thresholds=[
                tfma.PerSliceMetricThreshold(
                    slicing_specs=[tfma.SlicingSpec()],
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(lower_bound={'value': 0.6}),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={'value': -1e-10})))
            ]),
        })])

evaluator = tfx.components.Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config)

# Pusher solo publica el modelo si el Evaluator lo "bendice".
pusher = tfx.components.Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=tfx.proto.PushDestination(
        filesystem=tfx.proto.PushDestination.Filesystem(base_directory=serving_model_dir)))
```

Las `slicing_specs` son la pieza clave: en vez de calcular la precisión solo sobre todo el dataset, TFMA la recalcula además para cada especie de pingüino por separado, lo que te permite detectar si el modelo falla justo en un subgrupo que tu negocio no puede ignorar. Esta comparación continua entre versiones del modelo es lo que la documentación de TFX llama el **triángulo de insights de ML**: cuando un modelo falla, casi siempre es porque cambiaron los datos, el modelo se quedó corto, o cambiaron las condiciones del negocio.


## Metadata y procesamiento distribuido (concepto)

Dos piezas más completan el cuadro, aunque hoy no escribas código directamente contra ellas:

- **Metadata (MLMD, ML Metadata)**: cada ejecución de cada componente queda registrada —qué datos entraron, qué parámetros se usaron, qué artefacto salió— en un almacén respaldado por una base de datos (SQLite en nuestros ejemplos, MySQL en producción). Esto te da trazabilidad (qué datos entrenaron qué modelo), comparación entre ejecuciones y reutilización de resultados ya calculados (`cache`), en vez de recalcular pasos costosos que no cambiaron
- **Procesamiento distribuido con Apache Beam**: componentes como `ExampleGen`, `StatisticsGen` y `Transform` no procesan los datos directamente, delegan en Apache Beam, un modelo de programación que puede ejecutarse en tu portátil (el runner local que hemos usado hoy) o distribuirse sobre un clúster con Apache Spark, Apache Flink o Google Cloud Dataflow, sin cambiar el código de la transformación

Como desarrollador o desarrolladora casi nunca piensas en esta capa explícitamente: escribes tu `preprocessing_fn` una vez y TFX decide dónde y cómo ejecutarla.


## Resumen

- TFX lleva un modelo de un notebook a un sistema de producción fiable y repetible, organizado como un pipeline (DAG) de componentes
- El pipeline mínimo es `ExampleGen` + `Trainer` + `Pusher`; lo construiste y ejecutaste hoy de principio a fin
- TFDV (`StatisticsGen`, `SchemaGen`, `ExampleValidator`) valida que los datos nuevos sigan el esquema esperado antes de entrenar
- TFT (`Transform`) aplica la misma `preprocessing_fn` en entrenamiento y en servicio, evitando el sesgo entrenamiento-servicio
- TFMA (`Evaluator`) analiza el modelo por segmentos de datos y decide, comparando con la versión anterior, si el modelo nuevo debe desplegarse
- El metadata (MLMD) y Apache Beam son las capas horizontales que dan trazabilidad y escalabilidad al pipeline, casi siempre sin que tengas que tocarlas directamente

Ahora te toca a ti: abre el notebook de práctica de esta clase para extender este pipeline con tus propias manos.
